# Loading

In [ ]:
import omicverse as ov
import pandas as pd
import json
import numpy as np
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
ov.plot_set(font_path='Arial')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


# scRNA-seq data

In [ ]:
adata = sc.read_h5ad("Process_Data/to_do_list_2/obj_sc_20241104.h5ad")
adata

In [ ]:
cluster2annotation = {
    'Gaba': 'GABA neuron',
    'Glu': 'GLU neuron',
    'Immune cell': 'Microglia',
    'NPC': 'NPC',
    'Neuron': 'IPC',
    'Non-neuron': 'ChP',
    'OPC': 'OPC',
    'VLMC': 'VLMC',
}

adata.obs['annotation_major'] = adata.obs['annotation_major'].map(cluster2annotation).astype('category')
adata.uns['annotation_major_colors'] = ['#D1352B', '#7DBFA7', '#D2EBC8', '#3C77AF','#AECDE1','#EE934E','#9B5B33','#F5CFE4']

In [ ]:
output_dir = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_3/'

sc.settings.figdir = output_dir

In [ ]:
Gene_list = ['SST','VIP','PVALB','CCK','NPY','TAC1','PENK','POMC','TRH','GAL','PDYN']

axes = sc.pl.umap(
    adata,
    color=Gene_list,
    cmap='Reds',
    vmax='p99.99',
    ncols=4,
    show=False,
    hspace=0.3,
)

fig = axes[0].figure 
fig.set_size_inches(15, 10)

save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_3/umap_markers.pdf'
fig.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)

In [ ]:
Gene_list = ['CRYAB','NR4A1']

axes = sc.pl.umap(
    adata,
    color=Gene_list,
    cmap='Reds',
    vmax='p99.99',
    ncols=4,
    show=False,
    hspace=0.3,
)

fig = axes[0].figure 
fig.set_size_inches(9, 4)

save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_3/umap_markers_20251219.pdf'
fig.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)

In [ ]:
sc.pl.dotplot(
        adata,
        Gene_list,
        "annotation_major",
        dendrogram=False,
        standard_scale = 'var',
        cmap="RdBu_r",
        colorbar_title=f'Mean expression',
        figsize=(5, 3),
        save='dotplot_markers.pdf'
    )

# Spatial transcriptomics data

In [ ]:
gene_df = pd.read_csv("Process_Data/to_do_list_5/Table_S2.csv",index_col=[0])
gene_df

In [ ]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

Gene_list = ['SST', 'VIP', 'PVALB', 'CCK', 'NPY', 'TAC1', 'PENK', 'POMC', 'TRH', 'GAL', 'PDYN']

region_map = {
    'Hipp': 'HIP',
    'HB_Chp': 'Hindbrain_ChP',
    'EpD': 'ependymal region',
    'Medulla': 'Hindbrain',
    'SC/AON': 'subthalamic nucleus'
}

groups = gene_df['group'].unique()

# 初始化存储容器
results_total = {}
results_counts = []
results_props = []

os.makedirs("Figure/To_do_list_3", exist_ok=True)

for group_name in groups:
    region = region_map.get(group_name, group_name)
    adata_path = f"Process_Data/bin100_3D_region/combined_adata_{region}.h5ad"
        
    print(f"Calculating Region: {region} ...")
    
    adata = sc.read_h5ad(adata_path)
    
    n_total = adata.n_obs
    results_total[region] = n_total
    
    region_counts = {'region': region}
    region_props = {'region': region}
    
    for gene in Gene_list:
        if gene in adata.var_names:
            exp_data = adata[:, gene].X
            if hasattr(exp_data, "toarray"):
                count = (exp_data.toarray() > 0).sum()
            else:
                count = (exp_data > 0).sum()
        else:
            count = 0  # 如果基因不在该数据集中
            
        region_counts[gene] = count
        region_props[gene] = count / n_total if n_total > 0 else 0
        
    results_counts.append(region_counts)
    results_props.append(region_props)

df_total = pd.DataFrame.from_dict(results_total, orient='index', columns=['Total_Spots'])
df_counts = pd.DataFrame(results_counts).set_index('region')
df_props = pd.DataFrame(results_props).set_index('region')

# 保存为 CSV
df_total.to_csv("Process_Data/to_do_list_3/Matrix_Total_Spots_per_Region.csv")
df_counts.to_csv("Process_Data/to_do_list_3/Matrix_Gene_Counts_per_Region.csv")
df_props.to_csv("Process_Data/to_do_list_3/Matrix_Gene_Proportions_per_Region.csv")

print("Generate matrix file：Total_Spots, Gene_Counts, Gene_Proportions (CSV)")

In [ ]:
import matplotlib.pyplot as plt

colors_11 = [
    '#841f61', '#d357a0', '#f6bfd8', '#6c65ad', '#acdfea', 
    '#00b0b1', '#7e2e91', '#16ab4b', '#f9a232', '#ee3332', '#057996'
]

gene_color_map = dict(zip(Gene_list, colors_11))
for gene in Gene_list:
    plot_data = (df_props[gene].sort_values(ascending=False)) * 100
    plt.figure(figsize=(9, 6))
    ax = plot_data.plot(
        kind='bar', 
        color=gene_color_map[gene], 
        alpha=0.9, 
        edgecolor='black', 
        linewidth=1.5  # 加粗柱子边框
    )
    
    #plt.title(f'Proportion of Spots Expressing {gene} across Regions', fontsize=22, fontweight='bold', pad=20)
    plt.ylabel(f'{gene} expression proportion (%)', fontsize=13, )
    plt.xlabel(None) 
    
    plt.xticks(rotation=45, ha='right', fontsize=13, )
    plt.yticks(fontsize=13)
    
    for spine in ax.spines.values():
        spine.set_linewidth(2)    
    
    plt.tight_layout()
    
    save_path = f"Figure/To_do_list_3/{gene}_proportion_comparison.png"
    plt.savefig(save_path)

print("所有基因的可视化柱状图已保存在 Figure/To_do_list_3/ 目录下。")